In [1]:
import pandas as pd


## Kaggle

In [2]:
data = pd.read_csv("../data/raw/kaggle/full_data.csv")

In [3]:
pd.set_option("display.max_rows", None)

# which category was awarded from when to when
data["film_year"] = data["Year"].astype(str).str[:4].astype(int)

spans = (data.groupby("CanonicalCategory")["film_year"]
             .agg(first="min", last="max", years_awarded="nunique")
             .sort_values(["first", "last"]))
print(spans)

# all categories that have been awarded since the start
years_per_cat = data.groupby("CanonicalCategory")["film_year"].nunique()
n_years = data["film_year"].nunique()
always_there = years_per_cat[years_per_cat == n_years].index
print(list(always_there))

                                                    first  last  years_awarded
CanonicalCategory                                                             
DIRECTING (Comedy Picture)                           1927  1927              1
DIRECTING (Dramatic Picture)                         1927  1927              1
UNIQUE AND ARTISTIC PICTURE                          1927  1927              1
WRITING (Title Writing)                              1927  1927              1
WRITING (Original Story)                             1927  1956             27
SPECIAL AWARD                                        1927  2017             19
ACTOR IN A LEADING ROLE                              1927  2025             98
ACTRESS IN A LEADING ROLE                            1927  2025             98
ART DIRECTION                                        1927  2025             73
BEST PICTURE                                         1927  2025             98
CINEMATOGRAPHY                                      

In [4]:
# find the first year from which the most categories were awarded every year
max_count = 0
best_year = None

for from_year in range(1928, 2025):
    part = data[data["film_year"] >= from_year]
    years_per_cat = part.groupby("CanonicalCategory")["film_year"].nunique()
    always_there = years_per_cat[years_per_cat == part["film_year"].nunique()].index

    if len(always_there) > max_count:
        max_count = len(always_there)     
        best_year = from_year
        data_always = part[part["CanonicalCategory"].isin(always_there)]
        print(data_always["CanonicalCategory"].unique())

# from this year on, these categories were awarded without a gap 
print(best_year, max_count)

print(data_always["CanonicalCategory"].unique())

['ACTOR IN A LEADING ROLE' 'ACTRESS IN A LEADING ROLE' 'DIRECTING'
 'BEST PICTURE' 'WRITING (Adapted Screenplay)']
['ACTOR IN A LEADING ROLE' 'ACTRESS IN A LEADING ROLE' 'DIRECTING'
 'BEST PICTURE' 'SHORT FILM (Animated)' 'WRITING (Adapted Screenplay)']
['ACTOR IN A LEADING ROLE' 'ACTRESS IN A LEADING ROLE' 'DIRECTING'
 'FILM EDITING' 'MUSIC (Original Song)' 'BEST PICTURE'
 'SHORT FILM (Animated)' 'WRITING (Adapted Screenplay)']
['ACTOR IN A LEADING ROLE' 'ACTOR IN A SUPPORTING ROLE'
 'ACTRESS IN A LEADING ROLE' 'ACTRESS IN A SUPPORTING ROLE' 'DIRECTING'
 'FILM EDITING' 'MUSIC (Original Song)' 'BEST PICTURE'
 'SHORT FILM (Animated)' 'WRITING (Adapted Screenplay)']
['ACTOR IN A LEADING ROLE' 'ACTOR IN A SUPPORTING ROLE'
 'ACTRESS IN A LEADING ROLE' 'ACTRESS IN A SUPPORTING ROLE' 'DIRECTING'
 'DOCUMENTARY (Feature)' 'FILM EDITING' 'MUSIC (Original Song)'
 'BEST PICTURE' 'SHORT FILM (Animated)' 'WRITING (Adapted Screenplay)']
['ACTOR IN A LEADING ROLE' 'ACTOR IN A SUPPORTING ROLE'
 'ACTRE

In [5]:
# since which year each of these categories has been awarded every year

WANT_CATEGORIES = ['ACTOR IN A LEADING ROLE', 'ACTOR IN A SUPPORTING ROLE',
 'ACTRESS IN A LEADING ROLE', 'ACTRESS IN A SUPPORTING ROLE',
 'ANIMATED FEATURE FILM', 'CINEMATOGRAPHY', 'COSTUME DESIGN', 'DIRECTING',
 'DOCUMENTARY (Feature)', 'DOCUMENTARY (Short Subject)', 'FILM EDITING',
 'INTERNATIONAL FEATURE FILM', 'MAKEUP AND HAIRSTYLING',
 'MUSIC (Original Score)', 'MUSIC (Original Song)', 'BEST PICTURE',
 'VISUAL EFFECTS', 'WRITING (Adapted Screenplay)',
 'WRITING (Original Screenplay)']


years = sorted(data["film_year"].unique())
per_year = data.groupby("film_year")["CanonicalCategory"].apply(set)

since = {}
for cat in WANT_CATEGORIES:
    start = None
    for y in reversed(years):        
        if cat in per_year[y]:
            start = y
        else:
            break                    
    since[cat] = start

since = pd.Series(since).sort_values()
print(since)
print("All awarded without a gap since:", int(since.max()))

data_want = data[(data["film_year"] >= since.max()) &
                 (data["CanonicalCategory"].isin(WANT_CATEGORIES))]



# dataset size from that year on
print(len(data[data["film_year"] >= int(since.max())]))

ACTOR IN A LEADING ROLE          1927
ACTRESS IN A LEADING ROLE        1927
BEST PICTURE                     1927
WRITING (Adapted Screenplay)     1927
DIRECTING                        1928
MUSIC (Original Song)            1934
FILM EDITING                     1934
ACTOR IN A SUPPORTING ROLE       1936
ACTRESS IN A SUPPORTING ROLE     1936
DOCUMENTARY (Feature)            1947
WRITING (Original Screenplay)    1949
INTERNATIONAL FEATURE FILM       1954
MUSIC (Original Score)           1958
DOCUMENTARY (Short Subject)      1958
CINEMATOGRAPHY                   1967
COSTUME DESIGN                   1967
MAKEUP AND HAIRSTYLING           1984
VISUAL EFFECTS                   1991
ANIMATED FEATURE FILM            2001
dtype: int64
All awarded without a gap since: 2001
3352


In [6]:
ids = pd.read_csv("../data/interim/ids.csv")
print(ids["imdb_id"].isna().sum(), "of", len(ids), "articles without an IMDb ID")

0 of 1276 articles without an IMDb ID


## Analyze nominations

In [ ]:
nominations = pd.read_csv("../data/processed/nominations.csv")
nominations.info()
display(nominations[["Film","year_ceremony", "bafta_nom","n_nominees","oscar_total_noms","prev_oscar_noms","prev_oscar_wins","genres"]].head())

genres = nominations["genres"].dropna().str.split("|").explode()

print(genres.nunique(), "different genres")
print(genres.value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 621 entries, 0 to 620
Data columns (total 43 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year_ceremony       621 non-null    int64  
 1   year_film           621 non-null    int64  
 2   Ceremony            621 non-null    int64  
 3   Film                621 non-null    object 
 4   imdb_id             621 non-null    object 
 5   won                 621 non-null    int64  
 6   n_nominees          621 non-null    int64  
 7   oscar_total_noms    621 non-null    int64  
 8   has_director_nom    621 non-null    int64  
 9   has_editing_nom     621 non-null    int64  
 10  has_screenplay_nom  621 non-null    int64  
 11  n_acting_noms       621 non-null    int64  
 12  prev_oscar_noms     444 non-null    float64
 13  prev_oscar_wins     444 non-null    float64
 14  bafta_nom           201 non-null    float64
 15  cc_nom              226 non-null    float64
 16  dga_nom 

,Film,year_ceremony,bafta_nom,n_nominees,oscar_total_noms,prev_oscar_noms,prev_oscar_wins,genres
0,Wings,1928,NaN,3,2,NaN,NaN,Drama|Action|War|Romance
1,The Racket,1928,NaN,3,1,NaN,NaN,Crime|Drama
2,7th Heaven,1928,NaN,3,5,NaN,NaN,Drama|Romance
3,The Broadway Melody,1929,NaN,5,3,NaN,NaN,Music|Romance|Drama
4,Alibi,1929,NaN,5,3,NaN,NaN,Crime|Drama
5,In Old Arizona,1929,NaN,5,5,NaN,NaN,Western
6,Hollywood Revue,1929,NaN,5,1,NaN,NaN,Comedy|Music
7,The Patriot,1929,NaN,5,5,NaN,NaN,Drama|History
8,All Quiet on the Western Front,1930,NaN,5,4,NaN,NaN,Drama|War
9,The Big House,1930,NaN,5,4,NaN,NaN,Crime|Drama


17 different genres
genres
Drama              518
Romance            204
Comedy             130
History            106
Crime               73
Thriller            72
War                 65
Adventure           56
Music               37
Action              36
Mystery             30
Fantasy             28
Western             23
Family              23
Science Fiction     20
Horror               7
Animation            3
Name: count, dtype: int64
